In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from statsmodels.tsa.api import VAR

import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# TVP-VAR IRF 3D Surface + Event Mapping
# - Input: ./tvpvar_input_scaled.csv
# - Interactive selection: shock / response feature
# - x-axis = Date
# - y-axis = Horizon
# - z-axis = IRF
# ============================================================

# -----------------------------
# Config
# -----------------------------
INPUT_FILE = Path("./tvpvar_input_scaled.csv")
DATE_COL = "Date"

VAR_COLS = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

DEFAULT_SHOCK_VAR = "dlog_SOLVPN"
DEFAULT_RESPONSE_VAR = "dlog_COPPER"

LAG_ORDER = 1
ROLLING_WINDOW = 120
HORIZON = 20

MIN_REQUIRED_ROWS = ROLLING_WINDOW + LAG_ORDER + 5

# -----------------------------
# Event Mapping
# -----------------------------
EVENTS = [
    {"date": "2022-11-30", "label": "ChatGPT 출시", "group": "AI / Data Center", "color": "blue"},
    {"date": "2023-01-23", "label": "MSFT–OpenAI 투자/협력", "group": "AI / Data Center", "color": "blue"},
    {"date": "2023-03-14", "label": "GPT-4 공개", "group": "AI / Data Center", "color": "blue"},
    {"date": "2023-11-06", "label": "OpenAI DevDay", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-01-01", "label": "AI 투자 사이클 시작", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-03-18", "label": "NVIDIA Blackwell 발표", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-05-01", "label": "Hyperscaler capex 확대", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-07-31", "label": "BigTech AI capex 급증", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-10-15", "label": "OCP Blackwell 관련", "group": "AI / Data Center", "color": "blue"},
    {"date": "2025-01-21", "label": "Stargate 프로젝트", "group": "AI / Data Center", "color": "blue"},

    {"date": "2021-11-15", "label": "구리 공급 부족 신호", "group": "Copper / Supply", "color": "red"},
    {"date": "2023-03-01", "label": "구리 가격 상승 전망", "group": "Copper / Supply", "color": "red"},
    {"date": "2023-11-28", "label": "파나마 구리 광산 폐쇄", "group": "Copper / Supply", "color": "red"},
    {"date": "2024-08-15", "label": "LME 구리 재고 급감", "group": "Copper / Supply", "color": "red"},
    {"date": "2025-02-25", "label": "미국 구리 수입 조사 시작", "group": "Copper / Supply", "color": "red"},
    {"date": "2025-03-10", "label": "구리 정책 리스크 확대", "group": "Copper / Supply", "color": "red"},

    {"date": "2021-04-01", "label": "클라우드 수요 증가", "group": "Infra / System", "color": "green"},
    {"date": "2021-10-28", "label": "Meta 사명 변경", "group": "Infra / System", "color": "green"},
    {"date": "2022-03-22", "label": "H100 발표", "group": "Infra / System", "color": "green"},
    {"date": "2022-06-15", "label": "공급망 불안", "group": "Infra / System", "color": "green"},
    {"date": "2022-09-15", "label": "에너지 가격 급등", "group": "Infra / System", "color": "green"},
    {"date": "2022-09-20", "label": "H100 생산 시작", "group": "Infra / System", "color": "green"},
    {"date": "2023-05-01", "label": "AI 인프라 수요 증가", "group": "Infra / System", "color": "green"},
    {"date": "2023-07-01", "label": "GPU 부족", "group": "Infra / System", "color": "green"},
    {"date": "2023-09-15", "label": "데이터센터 딜 최대치", "group": "Infra / System", "color": "green"},
    {"date": "2023-10-15", "label": "액체 냉각 도입 논의", "group": "Infra / System", "color": "green"},
    {"date": "2024-09-20", "label": "전력 계약 이슈", "group": "Infra / System", "color": "green"},
]

# -----------------------------
# Helper
# -----------------------------
def compute_orth_irf_matrix(var_result, horizon: int) -> np.ndarray:
    irf_obj = var_result.irf(horizon)
    return irf_obj.orth_irfs  # shape: (horizon+1, k, k)

def build_irf_surface_figure(df, shock_var: str, response_var: str):
    shock_idx = VAR_COLS.index(shock_var)
    response_idx = VAR_COLS.index(response_var)

    dates_for_plot = []
    surface_rows = []

    for t in range(ROLLING_WINDOW, len(df)):
        window_df = df.iloc[t - ROLLING_WINDOW:t].copy()
        end_date = df.iloc[t][DATE_COL]
        y = window_df[VAR_COLS].copy()

        try:
            model = VAR(y)
            result = model.fit(LAG_ORDER)

            irf_mat = compute_orth_irf_matrix(result, HORIZON)
            pair_irf = irf_mat[:, response_idx, shock_idx]  # [horizon+1]

            surface_rows.append(pair_irf)
            dates_for_plot.append(end_date)

        except Exception:
            surface_rows.append(np.full(HORIZON + 1, np.nan))
            dates_for_plot.append(end_date)

    surface = np.array(surface_rows)  # [time, horizon+1]

    if surface.size == 0:
        raise ValueError("IRF surface가 생성되지 않았습니다.")

    plot_dates = pd.to_datetime(dates_for_plot)
    horizons = np.arange(HORIZON + 1)

    z = surface.T  # [horizon+1, time]

    z_min = np.nanmin(z)
    z_max = np.nanmax(z)

    plot_date_min = plot_dates.min()
    plot_date_max = plot_dates.max()

    fig = go.Figure()

    fig.add_trace(
        go.Surface(
            x=plot_dates,
            y=horizons,
            z=z,
            colorbar=dict(title="IRF"),
            name="IRF Surface",
            showscale=True
        )
    )

    for event in EVENTS:
        event_date = pd.to_datetime(event["date"])

        if not (plot_date_min <= event_date <= plot_date_max):
            continue

        fig.add_trace(
            go.Scatter3d(
                x=[event_date, event_date],
                y=[0, 0],
                z=[z_min, z_max],
                mode="lines+text",
                line=dict(color=event["color"], width=6),
                text=["", event["label"]],
                textposition="top center",
                name=event["group"],
                hovertemplate=(
                    f"Group: {event['group']}<br>"
                    f"Date: {event_date.strftime('%Y-%m-%d')}<br>"
                    f"Event: {event['label']}<extra></extra>"
                ),
                showlegend=False
            )
        )

    legend_groups = [
        ("AI / Data Center", "blue"),
        ("Copper / Supply", "red"),
        ("Infra / System", "green"),
    ]

    for group_name, color in legend_groups:
        fig.add_trace(
            go.Scatter3d(
                x=[None],
                y=[None],
                z=[None],
                mode="lines",
                line=dict(color=color, width=8),
                name=group_name,
                showlegend=True
            )
        )

    fig.update_layout(
        title=f"TVP-VAR IRF Surface: {shock_var} shock → {response_var} response",
        scene=dict(
            xaxis=dict(
                title="Date",
                autorange="reversed"
            ),
            yaxis=dict(
                title="Horizon"
            ),
            zaxis=dict(
                title="IRF"
            ),
            camera=dict(
                eye=dict(x=1.55, y=1.45, z=0.95)
            )
        ),
        width=1400,
        height=900,
        margin=dict(l=20, r=20, t=60, b=20)
    )

    return fig

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

use_cols = [DATE_COL] + VAR_COLS
df = df[use_cols].copy()
df = df.dropna().reset_index(drop=True)

if len(df) < MIN_REQUIRED_ROWS:
    raise ValueError(
        f"데이터가 너무 적습니다. 현재 행 수={len(df)}, "
        f"최소 필요 행 수는 대략 {MIN_REQUIRED_ROWS} 이상입니다."
    )

# -----------------------------
# Interactive widgets
# -----------------------------
shock_dropdown = widgets.Dropdown(
    options=VAR_COLS,
    value=DEFAULT_SHOCK_VAR,
    description="Shock:",
    layout=widgets.Layout(width="350px")
)

response_dropdown = widgets.Dropdown(
    options=VAR_COLS,
    value=DEFAULT_RESPONSE_VAR,
    description="Response:",
    layout=widgets.Layout(width="350px")
)

output = widgets.Output()

def update_plot(change=None):
    with output:
        clear_output(wait=True)

        shock_var = shock_dropdown.value
        response_var = response_dropdown.value

        if shock_var == response_var:
            print("Shock 변수와 Response 변수는 서로 달라야 합니다.")
            return

        fig = build_irf_surface_figure(df, shock_var, response_var)
        fig.show()

shock_dropdown.observe(update_plot, names="value")
response_dropdown.observe(update_plot, names="value")

display(widgets.HBox([shock_dropdown, response_dropdown]))
display(output)

update_plot()

Output()